In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

df = pd.read_excel('/content/oasis_cross-sectional-5708aa0a98d82080.xlsx')
B  = df[df['CDR'].notna()].reset_index(drop=True)
y  = B['CDR'].apply(lambda c: 0 if c == 0 else (1 if c == 0.5 else 2)).values
age = B[['Age']].values
P  = np.load('/content/oof_B_3class_1ch.npy')
cv = list(StratifiedKFold(5, shuffle=True, random_state=42).split(B, y))
auc = lambda p: roc_auc_score(y, p, multi_class='ovr', average='macro')

def stack(X, balanced=False):
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000,
                        class_weight='balanced' if balanced else None))
    return cross_val_predict(clf, X, y, cv=cv, method='predict_proba')

logP = np.log(P + 1e-6)
print(f'MRI alone (raw)                 {auc(P):.3f}')
print(f'Age alone                       {auc(stack(age)):.3f}')
print(f'Combine MRI only, no age        {auc(stack(logP)):.3f}')
print(f'Combine MRI + age               {auc(stack(np.column_stack([age, logP]))):.3f}')
print(f'Combine MRI + age, balanced     {auc(stack(np.column_stack([age, logP]), True)):.3f}')

fold_aucs = [roc_auc_score(y[te], P[te], multi_class='ovr', average='macro') for _, te in cv]
print(f'\nMRI per-fold AUCs: {np.round(fold_aucs, 3)}  mean {np.mean(fold_aucs):.3f}  vs pooled {auc(P):.3f}')